# Unit 25 · PyTorch — First Contact

**Learn with Adi — Python Programming (Bridge · Python for ML & DL)**

This is the one unit whose code cannot run on the study-notes page — PyTorch is far too big to download into a browser. So **this notebook is the lab**. Colab already has torch installed; every cell below really runs. Run a cell with **Shift+Enter**.

Study notes: https://aditya-402.github.io/learn-with-adi/series/python-programming/unit25.html

What you are here to learn: what the objects *are* and how the code *reads*. Why any of it makes a model better belongs to the Deep Learning stage.

In [ ]:
import torch
import torch.nn as nn

print("torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

## 25.1 · Tensors are NumPy arrays with two superpowers

Same shape, same indexing, same broadcasting, same `@`. The two additions: a tensor can live on a GPU, and it can remember how it was computed.

In [ ]:
import numpy as np

a = np.array([[1., 2.], [3., 4.]])
t = torch.tensor([[1., 2.], [3., 4.]])

print(a.shape, "  vs  ", t.shape)     # (2, 2)  vs  torch.Size([2, 2])
print(a * 2)
print(t * 2)
print("same numbers, different printing")

In [ ]:
# The array vocabulary, on a tensor. Nothing here is new - only the import is.
x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])

print(x)
print("shape:", x.shape)
print("rows:", x.shape[0], "columns:", x.shape[1])
print("row 0:", x[0])
print("column 2:", x[:, 2])
print("reshaped:", x.reshape(3, 2))
print("device:", x.device)            # cpu, unless you moved it

In [ ]:
# Practice - the two multiplications. Predict both before you run.
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[10, 20], [30, 40]])

print(a * b)      # element-wise: matching positions
print(a @ b)      # matrix multiply: row meets column

In [ ]:
# Practice - crossing the border in both directions.
arr = np.array([[1., 2., 3.]])
t = torch.from_numpy(arr)
back = t.numpy()

print(type(arr), arr.shape)
print(type(t), t.shape)
print(type(back), back.shape)

## 25.2 · requires_grad and .backward() — the breadcrumb trail

Mark a tensor with `requires_grad=True` and every operation done to it is recorded. `.backward()` walks that record in reverse and parks the answer in `.grad`.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()

print("x is still:", x)
print("x.grad    :", x.grad)     # tensor(6.) - the slope of x**2 is 2x, and 2*3 = 6

In [ ]:
# Practice - try other points, and a longer expression.
for value in [1.0, 2.0, 5.0]:
    x = torch.tensor(value, requires_grad=True)
    y = 3 * x ** 2 + 5 * x        # slope should be 6x + 5
    y.backward()
    print(f"x={value}  torch says {x.grad.item():.1f}  formula says {6 * value + 5:.1f}")

In [ ]:
# Practice - gradients ACCUMULATE. This is why the training loop calls zero_grad().
x = torch.tensor(3.0, requires_grad=True)

for round_no in range(3):
    y = x ** 2
    y.backward()
    print(f"after backward #{round_no + 1}, x.grad = {x.grad.item()}")

print("6, 12, 18 - each backward ADDED to the last. zero_grad() is what stops that.")

## 25.3 · nn.Module is just a class you can already read

Inheritance, `super().__init__()`, layers as attributes, one method called `forward`. Units 13–16, wearing a lab coat.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()                 # the Unit 15 rule, non-negotiable here
        self.hidden = nn.Linear(3, 4)      # 3 numbers in, 4 out
        self.output = nn.Linear(4, 1)      # 4 in, 1 out

    def forward(self, x):
        x = torch.relu(self.hidden(x))
        return self.output(x)

model = Net()
print(model)

In [ ]:
# The model called like a function - note model(x), never model.forward(x).
sample = torch.randn(2, 3)      # 2 samples, 3 features each
print("in :", sample.shape)
print("out:", model(sample).shape)

print()
for name, p in model.named_parameters():
    print(f"{name:15s} {tuple(p.shape)}")

In [ ]:
# Practice - build your own. Two hidden layers: 8 -> 5 -> 5 -> 2.
# Fill in the blanks, then print the model and push a batch of 4 through it.
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Linear(8, 5)
        # self.b = ...
        # self.c = ...

    def forward(self, x):
        x = torch.relu(self.a(x))
        # x = ...
        return x

my = MyNet()
print(my)
print(my(torch.randn(4, 8)).shape)

## 25.4 · The training loop, read like prose

Forward → loss → zero_grad → backward → step. Five lines, always that order. Here they run for real on data we invent, so you can watch the loss fall.

In [ ]:
torch.manual_seed(0)

# Invent some honest data: y = 2*f1 - 3*f2 + 1
X = torch.randn(200, 2)
y = X @ torch.tensor([[2.0], [-3.0]]) + 1.0

model = nn.Linear(2, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(200):
    pred = model(X)                 # forward
    loss = loss_fn(pred, y)         # how wrong
    optimizer.zero_grad()           # clear old gradients
    loss.backward()                 # work out the blame
    optimizer.step()                # nudge the knobs

    if epoch % 40 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")

print()
print("learned weights:", model.weight.detach().numpy().round(3))
print("learned bias   :", model.bias.detach().numpy().round(3))
print("we hid 2.0, -3.0 and 1.0 in the data - see how close it got")

In [ ]:
# Practice - break it on purpose, then fix it.
# 1) Comment out optimizer.zero_grad() and rerun. Watch the loss misbehave.
# 2) Put it back and set lr=5.0 instead. Watch it explode to nan.
# 3) Set lr=0.0001 and watch it barely move.
# The step size is the whole personality of a training run.

## 25.5 · Your build — a batch of data through a layer

Four samples, three features each, through one 3 → 1 layer. Read the shapes out loud after every line; most deep-learning bugs are a shape nobody checked.

In [ ]:
torch.manual_seed(0)

x = torch.randn(4, 3)        # 4 samples, 3 features each
layer = nn.Linear(3, 1)      # a 3 -> 1 layer, weights created for you
out = layer(x)               # one forward pass

print("x     :", x.shape)
print("weight:", layer.weight.shape)   # stored transposed: (1, 3)
print("bias  :", layer.bias.shape)
print("out   :", out.shape)
print()
print(out)

In [ ]:
# The same arithmetic, by hand, so nothing is mysterious.
one_sample = x[0]                       # 3 numbers
w = layer.weight[0]                     # 3 numbers
b = layer.bias[0]

by_hand = (one_sample * w).sum() + b
print("by hand :", by_hand.item())
print("by layer:", out[0, 0].item())
print("a neuron is a weighted sum plus a bias. That is the entire secret.")

In [ ]:
# Practice - end to end, all of today in one cell.
# A tiny model, a tiny dataset, five lines in a loop, and a falling loss.
torch.manual_seed(1)

X = torch.randn(50, 3)
y = X @ torch.tensor([[1.0], [0.0], [-2.0]]) + 0.5

class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(3, 1)

    def forward(self, x):
        return self.layer(x)

model = Tiny()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    pred = model(X)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 25 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")

print("learned:", model.layer.weight.detach().numpy().round(2), "hidden: [1, 0, -2]")

## Problem bank

The page versions of these run in NumPy and plain Python; here you can do them in torch instead.

1. **Two layers, three shapes** — 8 samples of 5 features through a 5→3 layer and a 3→1 layer. Print the shape after each. (`torch.ones(8, 5)` gets you started.)
2. **The gradient desk** — pick any `x`, build `y = x ** 2` with `requires_grad=True`, and check `x.grad` against `2x`. Then try `y = x ** 3`.
3. **Read the model, answer in print** — for a class with `nn.Linear(10, 6)`, `nn.Linear(6, 6)`, `nn.Linear(6, 2)`: how many features in, how many out, how many layers, and what shape does a batch of 4 come out as? Build it and check yourself with `print(model)`.
4. **Stretch — a shape checker** — write a plain-Python `FakeLinear` / `FakeNet` pair that pushes a `(rows, cols)` tuple through two layers and returns `"shape mismatch"` when the columns don't match. You will have rebuilt PyTorch's most famous error message.

**Where this leads:** the *Build an LLM from Scratch* series is written in exactly this vocabulary — tensors, a class with a `forward`, and a loop of five lines. Nothing new to learn before you start it; only more of the same, stacked higher.

In [ ]:
# your problem-bank workspace